# 🚀 AI Hub Google Colab GPU Bridge
### 100% Free 16GB Dedicated Tesla T4 GPU for Zero-OOM Video Diffusion & SDXL

Connects your Google Colab **NVIDIA Tesla T4 (15.8 GB VRAM)** directly to AI Hub via a secure, zero-setup Cloudflare tunnel.

### ⚡ 10-Second Quick Start:
1. Check GPU: **Runtime -> Change runtime type -> T4 GPU**
2. Run Cell 1 below (`Shift + Enter`)
3. Copy your unique `https://xxxx.trycloudflare.com` URL printed at the bottom
4. Paste it into AI Hub **Tab 3 -> Google Colab GPU Bridge** and click **⚡ Connect / Test GPU**!

In [ ]:
# ==============================================================================
# 🚀 AI HUB GOOGLE COLAB GPU BRIDGE (FASTAPI + ZERO-OOM DIFFUSERS + CLOUDFLARE TUNNEL)
# 100% Free 16GB Dedicated Tesla T4 GPU for True Video Diffusion & SDXL
# ==============================================================================

# 1. Clean previous instances if re-running
!pkill -9 -f uvicorn > /dev/null 2>&1
!pkill -9 -f cloudflared > /dev/null 2>&1

# 2. Install dependencies (Diffusers, FastAPI, Cloudflare)
!pip install -q fastapi uvicorn diffusers transformers accelerate torch pyngrok pillow imageio[ffmpeg] torchvision sentencepiece
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ Zero-OOM Engine Dependencies and Cloudflare Tunnel ready!")

# 3. Fetch or Build the Zero-OOM GPU Microservice
import os, base64

# Method A: Fetch latest verified server directly from GitHub
!curl -s -L -o colab_bridge_server.py https://raw.githubusercontent.com/ankush-dev-eng/ai-hub/main/colab_bridge_server.py

# Method B: Self-contained fallback using base64 (zero escape errors)
if not os.path.exists("colab_bridge_server.py") or os.path.getsize("colab_bridge_server.py") < 500:
    print("Writing embedded server fallback...")
    b64_code = b"""..."""
    with open("colab_bridge_server.py", "wb") as f:
        f.write(base64.b64decode(b64_code))

# 4. Launch Microservice and Cloudflare Tunnel (Proxy-Free + Health Gated)
import subprocess, time, re, sys, os, urllib.request

# Kill any lingering processes on port 8000
!fuser -k 8000/tcp > /dev/null 2>&1 || true
!pkill -9 -f uvicorn > /dev/null 2>&1 || true
!pkill -9 -f cloudflared > /dev/null 2>&1 || true
time.sleep(1)

# Test import directly in notebook kernel first
try:
    import colab_bridge_server
    print("✅ colab_bridge_server module verified!")
except Exception as e:
    import traceback
    print("❌ Import Error:")
    traceback.print_exc()

# Launch uvicorn via sys.executable with full logging
log_file = open("uvicorn.log", "w")
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "colab_bridge_server:app", "--host", "0.0.0.0", "--port", "8000", "--log-level", "info"],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

print("⏳ Initializing GPU Microservice... Waiting for health check on http://127.0.0.1:8000...")
proxy_opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
server_ready = False
for _ in range(40):
    if proc.poll() is not None:
        print(f"❌ Uvicorn exited prematurely with code {proc.returncode}!")
        with open("uvicorn.log", "r") as f:
            print(f.read())
        break
    try:
        with proxy_opener.open("http://127.0.0.1:8000/health", timeout=1) as res:
            if res.status == 200:
                server_ready = True
                print("✅ GPU Microservice is verified ONLINE on http://127.0.0.1:8000!")
                break
    except Exception:
        pass
    time.sleep(0.5)

if not server_ready and proc.poll() is None:
    print("⚠️ Server is taking longer than expected. Recent logs:")
    with open("uvicorn.log", "r") as f:
        print(f.read()[-500:])

# Clear proxy environment for cloudflared to connect directly to 127.0.0.1
cf_env = {k: v for k, v in os.environ.items() if not k.lower().endswith("_proxy")}
cf_env["NO_PROXY"] = "*"
cf_env["no_proxy"] = "*"

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    env=cf_env
)
print("🚀 Starting Cloudflare Tunnel... Generating public endpoint...")
for _ in range(45):
    line = tunnel.stderr.readline()
    if "trycloudflare.com" in line:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            print("\n" + "="*60)
            print("🎉 YOUR COLAB GPU BRIDGE URL IS READY:")
            print(f"👉  {url}  👈")
            print("="*60)
            print("Copy this link and paste it into AI Hub Tab 3 -> Google Colab GPU Bridge!")
            break
    time.sleep(0.5)


## 🌟 What More Can You Do With Your Free 16GB Colab GPU?

Google Colab's **NVIDIA Tesla T4 (15.8 GB VRAM)** has massive untapped horsepower. Here are 4 high-performance workloads you can run directly in this notebook:

### 1. 🤖 Uncensored Local LLM Inference (Qwen 2.5 7B / Llama 3.1 8B)
Run full 8-billion parameter models locally in 4-bit/8-bit with zero censorship, infinite tokens, and private weights.

### 2. 🎙️ Whisper Large-v3 Speech-to-Text Studio
Transcribe 1-hour audio podcasts or meetings into structured JSON/SRT in under 60 seconds with 99% accuracy.

### 3. 🔍 Real-ESRGAN 4K Super-Resolution Image Upscaling
Upscale diffusion outputs to 4096x4096 print-ready resolution with sharp micro-texture recovery.

### 4. 🎬 60 FPS RIFE Video Frame Interpolation
Interpolate 16fps or 24fps diffusion clips into ultra-fluid 60 FPS slow-motion cinema.

In [ ]:
# ==============================================================================
# BONUS CAPABILITY: WHISPER LARGE-V3 SPEECH-TO-TEXT DEMO ON 16GB T4 GPU
# ==============================================================================
# Uncomment below to transcribe audio with Whisper Large-v3 in seconds:
# !pip install -q transformers torch torchaudio accelerate
# import torch
# from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
# device = "cuda:0" if torch.cuda.is_available() else "cpu"
# model_id = "openai/whisper-large-v3"
# model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True, use_safetensors=True).to(device)
# processor = AutoProcessor.from_pretrained(model_id)
# pipe = pipeline("automatic-speech-recognition", model=model, tokenizer=processor.tokenizer, feature_extractor=processor.feature_extractor, torch_dtype=torch.float16, device=device)
# print("✅ Whisper Large-v3 loaded on T4 GPU! Ready to transcribe audio.")
